# Aggregate climate data in a year and in the vegetation period

This script aggregates climate data (minimum and maximum monthly temperatures, monthly precipitation) to yearly climate data and to climate data within the vegetation period(s) within a given year.

In [1]:
"""
Pseudocode:
functions:
- convert_doy_to_date()
- aggregate_climate_year()
- aggregate_climate_veg_period()
    take the climate in the veg period weighted by how many days of that month are included in the veg period
    in the southern hemisphere add the jan-march to the oct-dec climate

main:
for year in years:
    load monthly climate tifs of one year
    load vegetation period tif
    apply convert_doy_to_date() to vegetation period tif (OR apply this inside the aggregation functions?)
    apply aggregate_climate() to all 12 months of climate data -> yearly aggregate 
    apply aggregate_climate_veg_period() to 12 months of climate data -> veg period aggregate
    save yearly aggregate and veg period aggregate to bands in tif
"""

'\nPseudocode:\nfunctions:\n- convert_doy_to_date()\n- aggregate_climate_year()\n- aggregate_climate_veg_period()\n    take the climate in the veg period weighted by how many days of that month are included in the veg period\n    in the southern hemisphere add the jan-march to the oct-dec climate\n\nmain:\nfor year in years:\n    load monthly climate tifs of one year\n    load vegetation period tif\n    apply convert_doy_to_date() to vegetation period tif (OR apply this inside the aggregation functions?)\n    apply aggregate_climate() to all 12 months of climate data -> yearly aggregate \n    apply aggregate_climate_veg_period() to 12 months of climate data -> veg period aggregate\n    save yearly aggregate and veg period aggregate to bands in tif\n'

## Functions

In [1]:
import os
import numpy as np
import rasterio
from rasterio.windows import Window
from datetime import datetime, timedelta
import calendar
import tempfile
from rasterio.transform import from_origin
from rasterio.transform import xy
from pyproj import Transformer


create_hemisphere_masks() creates masks to differentiate the northern and southern hemisphere. This is relevant because in the south the vegetation period is in the "winter" months instead of "summer" months. That means the calculation for the north should be based on months Jan - Dec (1-12), for the south it should be based on July - June next year (7-6).

I created a small overlap of the masks at the equator because otherwise there is the issue that there are two rows of pixels with nodata values. Now a latitude below 0.1 is treated with the southern hemisphere calculation. It does not really matter at the equator bc the vegetation period is the whole year no matter the calculation.

In [2]:
def create_hemisphere_masks(src):
    """
    Create northern and southern hemisphere masks
    with the same shape as the raster grid.

    Parameters
    ----------
    src : rasterio dataset
        Open rasterio dataset defining the raster grid.

    Returns
    -------
    north_mask : np.ndarray
        Boolean array with shape (height, width).
    south_mask : np.ndarray
        Boolean array with shape (height, width).
    """

    height = src.height
    width = src.width

    # Pixel row/column indices
    rows, cols = np.indices(
        (height, width)
    )

    # Pixel centre coordinates in raster CRS
    xs, ys = xy(
        src.transform,
        rows,
        cols
    )

    # Ensure arrays have raster shape
    xs = np.asarray(xs).reshape(height, width)
    ys = np.asarray(ys).reshape(height, width)

    # Transform MODIS sinusoidal -> WGS84
    transformer = Transformer.from_crs(
        src.crs,
        "EPSG:4326",
        always_xy=True
    )

    _, lat = transformer.transform(
        xs,
        ys
    )

    lat = np.asarray(lat).reshape(height, width)

    # Hemisphere masks
    north_mask = lat >= 0
    south_mask = lat <= 0.1 # a small overlap solves the nodata issue at equator

    return north_mask, south_mask

get_vegetation_calendar_dates() returns the actual dates for the 12 calendar months corresponding to the GAM vegetation year definition. It is used in the next function to convert relative doys in the vegetation year to actual dates.

In [3]:
def get_vegetation_calendar_dates(hemisphere, year):
    """
    Create actual dates for the vegetation year.

    The returned arrays contain the 12 calendar months
    corresponding to the GAM vegetation year definition.

    North:
        Jan-Dec of year

    South:
        Jul-Dec of year + Jan-Jun of year+1

    Returns
    -------
    dates : list(datetime)
        First day of each vegetation month
    """

    if hemisphere == "north":

        months = list(range(1, 13))
        years = [year] * 12

    elif hemisphere == "south":

        months = list(range(7, 13)) + list(range(1, 7))
        years = [year] * 6 + [year + 1] * 6

    else:
        raise ValueError(
            "hemisphere must be 'north' or 'south'"
        )

    dates = [
        datetime(y, m, 1)
        for y, m in zip(years, months)
    ]

    return dates

relative_doy_to_date() converts a day of year (doy) relative to the vegetation year (Jan - Dec for northern hemisphere or July - June next year for the southern hemisphere) back into actual dates in datetime format.

In [4]:
def relative_doy_to_date(relative_doy, hemisphere, year):
    """
    Convert vegetation-relative DOY into calendar dates.

    Parameters
    ----------
    relative_doy : ndarray
        DOY relative to vegetation-year start.

    Returns
    -------
    dates : ndarray
        Array of datetime objects.
    """

    start_dates = get_vegetation_calendar_dates(
        hemisphere,
        year
    )

    start = start_dates[0]

    dates = np.empty(
        relative_doy.shape,
        dtype=object
    )

    dates[:] = None

    valid = np.isfinite(relative_doy)

    for idx in zip(*np.where(valid)):

        dates[idx] = (
            start
            +
            timedelta(
                days=int(relative_doy[idx]) - 1
            )
        )

    return dates

month_overlap_fraction() calculates the fraction of days of a month that lie in a given vegetation period (within the onset date - end date). E.g. if the vegetation period onset is 16th March, the overlap fraction is ca. 0.5, i.e. half of the days of March lie in the vegetation period. This is needed to aggregate the climate data in the vegetation period, considering e.g. half of the precipitation in March or giving the March temperatures only half of the weight when averaging temperatures.

In [5]:
def month_overlap_fraction(
        month_year,
        month,
        onset_date,
        end_date
):
    """
    Calculate fraction of a month contained
    in the vegetation period.

    Parameters
    ----------
    onset_date : ndarray of datetime
        Vegetation onset dates per pixel.

    end_date : ndarray of datetime
        Vegetation end dates per pixel.

    Returns
    -------
    ndarray
        Fraction of each month included in vegetation period.
    """

    shape = onset_date.shape

    overlap_fraction = np.zeros(
        shape,
        dtype="float32"
    )

    days_in_month = calendar.monthrange(
        month_year,
        month
    )[1]

    month_start = datetime(
        month_year,
        month,
        1
    )

    month_end = datetime(
        month_year,
        month,
        days_in_month
    )


    valid = (
        onset_date != None
    ) & (
        end_date != None
    )


    for idx in zip(
        *np.where(valid)
    ):

        overlap_start = max(
            onset_date[idx],
            month_start
        )

        overlap_end = min(
            end_date[idx],
            month_end
        )

        overlap_days = (
            overlap_end
            -
            overlap_start
        ).days + 1

        
        overlap_fraction[idx] = (
            max(overlap_days, 0)
            /
            days_in_month
        )

    return overlap_fraction

calculate_calendar_year_vegetation_length() calculates the total length of the vegetation period(s) within one year. E.g. for the southern hemisphere it adds up the vegetation period in Jan-March with the one in Oct-Dec of the same year, e.g. year 2002. Maybe we need the length later as comparison point bc the precipitation is summed over that many days, so more days mean more precipitation. 

In [6]:
def calculate_calendar_year_vegetation_length(
        onset_prev,
        end_prev,
        onset_current,
        end_current,
        year
):
    """
    Calculate vegetation period length per pixel within a calendar year.

    Southern hemisphere:
        - January-June from previous vegetation-year raster
        - July-December from current vegetation-year raster

    Parameters
    ----------
    onset_prev, end_prev : ndarray
        Vegetation period dates from year-1 raster.

    onset_current, end_current : ndarray
        Vegetation period dates from year raster.

    year : int
        Calendar year.

    Returns
    -------
    veg_length : ndarray
        Vegetation days within calendar year per pixel.
    """
    # Convert datetime object arrays to numpy.datetime64[D]
    onset_prev = onset_prev.astype("datetime64[D]")
    end_prev = end_prev.astype("datetime64[D]")

    onset_current = onset_current.astype("datetime64[D]")
    end_current = end_current.astype("datetime64[D]")

    shape = onset_current.shape

    veg_length = np.zeros(
        shape,
        dtype="float32"
    )

    # -------------------------------------------------------
    # Define calendar-year boundaries
    # -------------------------------------------------------

    year_start = np.datetime64(
        f"{year}-01-01"
    )

    year_end = np.datetime64(
        f"{year}-12-31"
    )


    # -------------------------------------------------------
    # Previous vegetation period contribution
    # January-June
    # -------------------------------------------------------

    valid_prev = (
        np.isfinite(onset_prev)
        &
        np.isfinite(end_prev)
    )


    if np.any(valid_prev):

        overlap_start = np.maximum(
            onset_prev,
            year_start
        )

        overlap_end = np.minimum(
            end_prev,
            year_end
        )

        days_prev = (
            overlap_end - overlap_start
        ).astype("timedelta64[D]").astype(float) + 1


        days_prev = np.maximum(
            days_prev,
            0
        )

        veg_length[valid_prev] += days_prev[valid_prev]


    # -------------------------------------------------------
    # Current vegetation period contribution
    # July-December
    # -------------------------------------------------------

    valid_current = (
        np.isfinite(onset_current)
        &
        np.isfinite(end_current)
    )


    if np.any(valid_current):

        overlap_start = np.maximum(
            onset_current,
            year_start
        )

        overlap_end = np.minimum(
            end_current,
            year_end
        )

        days_current = (
            overlap_end - overlap_start
        ).astype("timedelta64[D]").astype(float) + 1


        days_current = np.maximum(
            days_current,
            0
        )

        veg_length[valid_current] += days_current[valid_current]


    return veg_length

read_band_names() is a helpfer function to read raster band names by their band description, not their number, because this makes the code less error prone.

In [7]:
def read_band_by_name(src, band_name, window):
    """
    Read a raster band by its description/name.
    """

    try:
        band_index = src.descriptions.index(band_name) + 1
    except ValueError:
        raise ValueError(
            f"Band '{band_name}' not found. "
            f"Available bands: {src.descriptions}"
        )

    return src.read(
        band_index,
        window=window
    ).astype("float32")

aggregate_climate_year() aggregates the climate data within a whole year:
- minimum and maximum monthly temperature:
    - mean, weighted by the number of days each month has
    - absolute minimum value found in any month
    - absolute maximum value found in any month
- monthly precipitation
    - summed across all months

In [8]:
def aggregate_climate_year(
        climate_files,
        year,
        output_file,
        nodata_value=-9999
):
    """
    Aggregate monthly climate rasters to yearly statistics.

    Temperature:
        - converts scale factor 0.1 to °C
        - calculates day-weighted mean e.g. (temp jan * 31 days + temp feb * 28 days) / (31 days + 28 days)
        - calculates min/max of monthly values

    Precipitation:
        - sums monthly precipitation

    Parameters
    ----------
    climate_files : list[str]
        Paths to 12 monthly climate GeoTIFFs.
        Bands:
            1: pr 
            2: tmmn
            3: tmmx

    year : int
        Calendar year of the aggregation.

    output_file : str
        Output GeoTIFF.

    nodata_value : int
        Input nodata value.
    """

    # -------------------------------------------------------
    # Open all climate rasters once
    # -------------------------------------------------------

    climate_srcs = [
        rasterio.open(f)
        for f in climate_files
    ]

    src0 = climate_srcs[0]

    profile = src0.profile.copy()

    profile.update(
        count=7,
        dtype="float32",
        nodata=np.nan
    )


    # Output bands:
    #
    # 1 tmmn_mean
    # 2 tmmx_mean
    # 3 tmmn_min
    # 4 tmmn_max
    # 5 tmmx_min
    # 6 tmmx_max
    # 7 pr_sum


    with rasterio.open(
        output_file,
        "w",
        **profile
    ) as dst:


        # Process raster block-wise
        for _, window in src0.block_windows(1):

            shape = (
                window.height,
                window.width
            )


            # -----------------------------------------------
            # Accumulators
            # -----------------------------------------------

            tmmn_weighted_sum = np.zeros(
                shape,
                dtype="float32"
            )

            tmmx_weighted_sum = np.zeros(
                shape,
                dtype="float32"
            )

            weight_sum = np.zeros(
                shape,
                dtype="float32"
            )

            pr_sum = np.zeros(
                shape,
                dtype="float32"
            )


            tmmn_min = np.full(
                shape,
                np.inf,
                dtype="float32"
            )

            tmmn_max = np.full(
                shape,
                -np.inf,
                dtype="float32"
            )

            tmmx_min = np.full(
                shape,
                np.inf,
                dtype="float32"
            )

            tmmx_max = np.full(
                shape,
                -np.inf,
                dtype="float32"
            )


            # -----------------------------------------------
            # Loop over months
            # -----------------------------------------------

            for month, src in enumerate(
                climate_srcs,
                start=1
            ):

                tmmn = read_band_by_name(
                    src,
                    "tmmn",
                    window
                )

                tmmx = read_band_by_name(
                    src,
                    "tmmx",
                    window
                )

                pr = read_band_by_name(
                    src,
                    "pr",
                    window
                )

                # Convert nodata to NaN
                tmmn[tmmn == nodata_value] = np.nan
                tmmx[tmmx == nodata_value] = np.nan
                pr[pr == nodata_value] = np.nan


                # Apply temperature scale factor
                tmmn *= 0.1
                tmmx *= 0.1


                # Number of days in this month
                days = calendar.monthrange(
                    year,
                    month
                )[1]


                # -------------------------------------------
                # Temperature weighted mean
                # -------------------------------------------

                valid_temp = (
                    np.isfinite(tmmn)
                    &
                    np.isfinite(tmmx)
                )


                tmmn_weighted_sum[valid_temp] += (
                    tmmn[valid_temp] * days
                )

                tmmx_weighted_sum[valid_temp] += (
                    tmmx[valid_temp] * days
                )

                weight_sum[valid_temp] += days


                # -------------------------------------------
                # Precipitation sum
                # -------------------------------------------

                valid_pr = np.isfinite(pr)

                pr_sum[valid_pr] += pr[valid_pr]


                # -------------------------------------------
                # Temperature min/max
                # -------------------------------------------

                tmmn_min = np.fmin(
                    tmmn_min,
                    tmmn
                )

                tmmn_max = np.fmax(
                    tmmn_max,
                    tmmn
                )

                tmmx_min = np.fmin(
                    tmmx_min,
                    tmmx
                )

                tmmx_max = np.fmax(
                    tmmx_max,
                    tmmx
                )


            # -----------------------------------------------
            # Finalize means
            # -----------------------------------------------

            with np.errstate(
                divide="ignore",
                invalid="ignore"
            ):

                tmmn_mean = (
                    tmmn_weighted_sum
                    /
                    weight_sum
                )

                tmmx_mean = (
                    tmmx_weighted_sum
                    /
                    weight_sum
                )


            # Pixels without data
            no_data = weight_sum == 0

            outputs = {
                "tmmn_mean": tmmn_mean,
                "tmmx_mean": tmmx_mean,
                "tmmn_min": tmmn_min,
                "tmmn_max": tmmn_max,
                "tmmx_min": tmmx_min,
                "tmmx_max": tmmx_max,
                "pr_sum": pr_sum
            }


            for arr in outputs.values():
                arr[no_data] = np.nan

            dst.write(
                np.stack(list(outputs.values())).astype("float32"),
                window=window
            )

            for band, name in enumerate(outputs.keys(), start=1):
                dst.set_band_description(
                    band,
                    name
                )

    # Close raster handles
    for src in climate_srcs:
        src.close()

aggregate_climate_veg_period() aggregates climate data within the vegetation period(s) within one year. E.g. for the southern hemisphere it takes the climate in Jan - March (the vegetation period for that is stored in the vegetation period file of the previous year) and also Oct - Dec (the vegetation period for that is stored in the file of the current year) of the same year. This is important because we want to predict yearly NPP. It calculates:
- minimum and maximum monthly temperature:
    - mean in the vegetation period, weighted by the number of days each month has; each month contributes a weight corresponding to it's fraction that lies in the vegetation period
    - absolute minimum value found in any month in the vegetation period
    - absolute maximum value found in any month in the vegetation period
- monthly precipitation
    - summed across all months in the vegetation period; each month contributes a weight corresponding to it's fraction that lies in the vegetation period

A limitation of both the yearly(?) and vegetation period aggregation is that we assume the temperatures and precipitation to be uniformly distributed within each month, which is unlikely in reality.

In [9]:
def aggregate_climate_veg_period(
        climate_files,
        vegetation_files,
        year,
        hemisphere,
        mask,
        output_file,
        nodata_value=-9999
):
    """
    Aggregate climate variables over the vegetation period.

    Temperature:
        - day-weighted mean over vegetation period
        - minimum and maximum monthly values

    Precipitation:
        - proportional sum based on fraction of month
          inside vegetation period

    Parameters
    ----------
    climate_files : list[str]
        12 monthly climate rasters for the target year.

    vegetation_files : list[str]
        2 Vegetation rasters with bands:
            1 vegetation_onset
            2 vegetation_end
            3 vegetation_length

        Onset/end are relative DOY within vegetation year.

    year : int
        Output year.

    hemisphere : str
        "north" or "south"

    output_file : str
        Output GeoTIFF.
    """


    # -------------------------------------------------------
    # Open input rasters once
    # -------------------------------------------------------

    climate_srcs = [
        rasterio.open(f)
        for f in climate_files
    ]

    veg_srcs = [
        rasterio.open(v)
        for v in vegetation_files
    ]

    src0 = climate_srcs[0]


    profile = src0.profile.copy()

    profile.update(
        count=8,
        dtype="float32",
        nodata=np.nan
    )


    # output bands:
    #
    # 1 veg_tmmn_mean
    # 2 veg_tmmx_mean
    # 3 veg_tmmn_min
    # 4 veg_tmmn_max
    # 5 veg_tmmx_min
    # 6 veg_tmmx_max
    # 7 veg_pr_sum
    # 8 vegetation_length


    with rasterio.open(
        output_file,
        "w",
        **profile
    ) as dst:


        for _, window in src0.block_windows(1):

            shape = (
                window.height,
                window.width
            )

            mask_window = mask[
                window.row_off:window.row_off + window.height,
                window.col_off:window.col_off + window.width
            ]


            # ------------------------------------------------
            # Read vegetation information
            # ------------------------------------------------

            # TODO: read 2 vegetation files
            onset_prev = veg_srcs[0].read(
                1,
                window=window
            ).astype("float32")

            end_prev = veg_srcs[0].read(
                2,
                window=window
            ).astype("float32")

            length_prev = veg_srcs[0].read(
                3,
                window=window
            ).astype("float32")


            valid_veg_prev = (
                np.isfinite(onset_prev)
                &
                np.isfinite(end_prev)
                &
                (length_prev > 0)
            )

            onset_current = veg_srcs[1].read(
                1,
                window=window
            ).astype("float32")
            
            end_current = veg_srcs[1].read(
                2,
                window=window
            ).astype("float32")

            length_current = veg_srcs[1].read(
                3,
                window=window
            ).astype("float32")


            valid_veg_current = (
                np.isfinite(onset_current)
                &
                np.isfinite(end_current)
                &
                (length_current > 0)
            )

            valid_veg = (
                (
                    valid_veg_prev
                    |
                    valid_veg_current
                )
                &
                mask_window
            )
           
            
            # ------------------------------------------------
            # Convert relative DOY to calendar dates
            # ------------------------------------------------

            onset_date_prev = relative_doy_to_date(
                onset_prev,
                hemisphere,
                year-1
            )

            end_date_prev = relative_doy_to_date(
                end_prev,
                hemisphere,
                year-1
            )

            onset_date_current = relative_doy_to_date(
                onset_current,
                hemisphere,
                year
            )

            end_date_current = relative_doy_to_date(
                end_current,
                hemisphere,
                year
            )

            # ------------------------------------------------
            # Accumulators
            # ------------------------------------------------

            tmmn_sum = np.zeros(
                shape,
                dtype="float32"
            )

            tmmx_sum = np.zeros(
                shape,
                dtype="float32"
            )

            weight_sum = np.zeros(
                shape,
                dtype="float32"
            )

            pr_sum = np.zeros(
                shape,
                dtype="float32"
            )


            tmmn_min = np.full(
                shape,
                np.inf,
                dtype="float32"
            )

            tmmn_max = np.full(
                shape,
                -np.inf,
                dtype="float32"
            )

            tmmx_min = np.full(
                shape,
                np.inf,
                dtype="float32"
            )

            tmmx_max = np.full(
                shape,
                -np.inf,
                dtype="float32"
            )


            # ------------------------------------------------
            # Loop through months
            # ------------------------------------------------

            for month, src in enumerate(
                climate_srcs,
                start=1
            ):

                tmmn = read_band_by_name(
                    src,
                    "tmmn",
                    window
                )

                tmmx = read_band_by_name(
                    src,
                    "tmmx",
                    window
                )

                pr = read_band_by_name(
                    src,
                    "pr",
                    window
                )

                # nodata
                tmmn[tmmn == nodata_value] = np.nan
                tmmx[tmmx == nodata_value] = np.nan
                pr[pr == nodata_value] = np.nan


                # temperature scaling
                tmmn *= 0.1
                tmmx *= 0.1

                
                # ------------------------------------------------
                # Calculate overlap fraction per pixel
                # ------------------------------------------------

                overlap_fraction_prev = month_overlap_fraction(
                                    month_year=year,
                                    month=month,
                                    onset_date=onset_date_prev,
                                    end_date=end_date_prev
                                )

                overlap_fraction_current = month_overlap_fraction(
                    month_year=year,
                    month=month,
                    onset_date=onset_date_current,
                    end_date=end_date_current
                )

                overlap_fraction = overlap_fraction_prev + overlap_fraction_current

                valid = (
                    mask_window
                    &
                    (overlap_fraction > 0)
                    &
                    np.isfinite(tmmn)
                    &
                    np.isfinite(tmmx)
                )

                # ------------------------------------------------
                # Temperature weighted by included days
                # ------------------------------------------------
                days_month = calendar.monthrange(year, month)[1]

                overlap_days = (
                    overlap_fraction
                    *
                    days_month
                )

                tmmn_sum[valid] += (
                    tmmn[valid]
                    *
                    overlap_days[valid]
                )

                tmmx_sum[valid] += (
                    tmmx[valid]
                    *
                    overlap_days[valid]
                )


                weight_sum[valid] += (
                    overlap_days[valid]
                )

                # ------------------------------------------------
                # Precipitation summed by included days
                # ------------------------------------------------
                valid_pr = (
                    mask_window
                    &
                    (overlap_fraction > 0)
                    &
                    np.isfinite(pr)
                )

                pr_sum[valid_pr] += (
                    pr[valid_pr]
                    *
                    overlap_fraction[valid_pr]
                )


                # ------------------------------------------------
                # Monthly extremes within vegetation period
                # ------------------------------------------------

                tmmn_min[valid] = np.fmin(
                    tmmn_min[valid],
                    tmmn[valid]
                )

                tmmn_max[valid] = np.fmax(
                    tmmn_max[valid],
                    tmmn[valid]
                )

                tmmx_min[valid] = np.fmin(
                    tmmx_min[valid],
                    tmmx[valid]
                )

                tmmx_max[valid] = np.fmax(
                    tmmx_max[valid],
                    tmmx[valid]
                )



            # ------------------------------------------------
            # Finalize
            # ------------------------------------------------

            with np.errstate(
                divide="ignore",
                invalid="ignore"
            ):

                tmmn_mean = (
                    tmmn_sum /
                    weight_sum
                )

                tmmx_mean = (
                    tmmx_sum /
                    weight_sum
                )


            # Total length of vegetation period(s) in given year
            veg_length = calculate_calendar_year_vegetation_length(onset_date_prev, end_date_prev, onset_date_current, end_date_current, year)
            veg_length[~mask_window] = np.nan

            no_data = (
                ~valid_veg
                |
                (weight_sum == 0)
            )


            outputs = {
                "veg_tmmn_mean": tmmn_mean,
                "veg_tmmx_mean": tmmx_mean,
                "veg_tmmn_min": tmmn_min,
                "veg_tmmn_max": tmmn_max,
                "veg_tmmx_min": tmmx_min,
                "veg_tmmx_max": tmmx_max,
                "veg_pr_sum": pr_sum,
                "vegetation_length": veg_length
            }
            

            for arr in outputs.values():
                arr[no_data] = np.nan


            dst.write(
                np.stack(list(outputs.values()))
                .astype("float32"),
                window=window
            )

            for band, name in enumerate(outputs.keys(), start=1):
                dst.set_band_description(
                    band,
                    name
                )


    # close files
    for src in climate_srcs:
        src.close()

    for veg_src in veg_srcs:
        veg_src.close()

## Main

The main code aggregates the climate data for each year. It also aggregates for each year, the climate in the vegetation period(s) in that year. Because the doy to date conversion is different for northen and southern hemisphere, the function is run two times seperately where it calculates the climate aggregation for the north while the south pixels are masked and vice versa. In the end the two rasters are combined to one global raster.

For 2001 we do not have a vegetation period estimate for the southern hemisphere from Jan - June 2001, so I assumed it would be the same as Jan - June 2002. For 2024 we don't have an estimate of the vegetation period in the southern hemisphere from July - Dec 2024, so I assumed it would be the same as July - Dec 2023.

In [10]:
# -------------------------------------------------------
# Paths
# -------------------------------------------------------

#climate_path = "/Volumes/TOSHIBA EXT/non-equilibrium/data/climate_monthly"
#vegetation_path = "/Volumes/TOSHIBA EXT/non-equilibrium/data/vegetation_period"
#output_path = "/Volumes/TOSHIBA EXT/non-equilibrium/data/climate_yearly"
climate_path = "/Users/Wanja/Documents/non-equilibrium_data/climate_monthly"
vegetation_path = "/Users/Wanja/Documents/non-equilibrium_data/vegetation_period_new"
output_path = "/Users/Wanja/Documents/non-equilibrium_data/climate_yearly_new"

years = range(2002, 2024)


# -------------------------------------------------------
# Main loop
# -------------------------------------------------------

for year in years:

    print(f"\nProcessing {year}")


    # ---------------------------------------------------
    # Monthly climate files
    # ---------------------------------------------------

    climate_files = []

    for month in range(1, 13):

        climate_files.append(
            os.path.join(
                climate_path,
                f"Climate_monthly_{year}-{month:02d}-01.tif"
            )
        )


    # ---------------------------------------------------
    # Vegetation files
    # ---------------------------------------------------

    vegetation_previous = os.path.join(
        vegetation_path,
        f"vegetation_period_{year-1}.tif"
    )

    vegetation_current = os.path.join(
        vegetation_path,
        f"vegetation_period_{year}.tif"
    )


    # ---------------------------------------------------
    # Create hemisphere masks
    # ---------------------------------------------------

    with rasterio.open(climate_files[0]) as src:

        north_mask, south_mask = create_hemisphere_masks(src)


    # ---------------------------------------------------
    # Aggregate yearly climate
    # ---------------------------------------------------

    aggregate_climate_year(

        climate_files=climate_files,

        year=year,

        output_file=os.path.join(
            output_path,
            f"Climate_yearly_{year}.tif"
        )

    )


    # ---------------------------------------------------
    # Temporary hemisphere outputs
    # ---------------------------------------------------

    north_output = os.path.join(
        output_path,
        f"_north_{year}.tif"
    )

    south_output = os.path.join(
        output_path,
        f"_south_{year}.tif"
    )


    # ---------------------------------------------------
    # Northern hemisphere
    # ---------------------------------------------------

    aggregate_climate_veg_period(

        climate_files=climate_files,

        vegetation_files=[
            vegetation_previous,
            vegetation_current
        ],

        year=year,

        hemisphere="north",

        mask=north_mask,

        output_file=north_output

    )


    # ---------------------------------------------------
    # Southern hemisphere
    # ---------------------------------------------------

    aggregate_climate_veg_period(

        climate_files=climate_files,

        vegetation_files=[
            vegetation_previous,
            vegetation_current
        ],

        year=year,

        hemisphere="south",

        mask=south_mask,

        output_file=south_output

    )


    # ---------------------------------------------------
    # Merge hemispheres
    # ---------------------------------------------------

    final_output = os.path.join(
        output_path,
        f"Climate_vegetation_period_{year}.tif"
    )


    with rasterio.open(north_output) as north_src, \
         rasterio.open(south_output) as south_src:


        profile = north_src.profile.copy()


        with rasterio.open(
            final_output,
            "w",
            **profile
        ) as dst:
            # Preserve band names
            for i, desc in enumerate(
                north_src.descriptions,
                start=1
            ):
                dst.set_band_description(
                    i,
                    desc
                )

            for _, window in north_src.block_windows(1):

                north = north_src.read(
                    window=window
                )

                south = south_src.read(
                    window=window
                )


                mask_window = north_mask[
                    window.row_off:
                    window.row_off + window.height,

                    window.col_off:
                    window.col_off + window.width
                ]


                output = south.copy()

                output[:, mask_window] = (
                    north[:, mask_window]
                )


                dst.write(
                    output,
                    window=window
                )


    # ---------------------------------------------------
    # Remove temporary files
    # ---------------------------------------------------

    os.remove(north_output)
    os.remove(south_output)


print("\nFinished.")


Processing 2002


/var/folders/v8/1_2rlmrd0wl5w5t5n60h8s440000gn/T/ipykernel_59109/1236317376.py:14: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  return src.read(
/var/folders/v8/1_2rlmrd0wl5w5t5n60h8s440000gn/T/ipykernel_59109/3899900946.py:108: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  onset_prev = veg_srcs[0].read(
/var/folders/v8/1_2rlmrd0wl5w5t5n60h8s440000gn/T/ipykernel_59109/3899900946.py:113: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  end_prev = veg_srcs[0].read(
/var/folders/v8/1_2rlmrd0wl5w5t5n60h8s440000gn/T/ipykernel_59109/3899900946.py:118: DeprecationWarning: Setting the shape on a


Processing 2003

Processing 2004

Processing 2005

Processing 2006

Processing 2007

Processing 2008

Processing 2009

Processing 2010

Processing 2011

Processing 2012

Processing 2013

Processing 2014

Processing 2015

Processing 2016

Processing 2017

Processing 2018

Processing 2019

Processing 2020

Processing 2021

Processing 2022

Processing 2023

Finished.


## Tests

In [69]:
get_vegetation_calendar_dates("south", 2001)

[datetime.datetime(2001, 7, 1, 0, 0),
 datetime.datetime(2001, 8, 1, 0, 0),
 datetime.datetime(2001, 9, 1, 0, 0),
 datetime.datetime(2001, 10, 1, 0, 0),
 datetime.datetime(2001, 11, 1, 0, 0),
 datetime.datetime(2001, 12, 1, 0, 0),
 datetime.datetime(2002, 1, 1, 0, 0),
 datetime.datetime(2002, 2, 1, 0, 0),
 datetime.datetime(2002, 3, 1, 0, 0),
 datetime.datetime(2002, 4, 1, 0, 0),
 datetime.datetime(2002, 5, 1, 0, 0),
 datetime.datetime(2002, 6, 1, 0, 0)]

In [70]:
dates_north = relative_doy_to_date(np.array([ 15,  46,  74, 105, 135, 166, 196, 227, 258, 288, 319, 349]), "north", 2001)
print(dates_north)

[datetime.datetime(2001, 1, 15, 0, 0) datetime.datetime(2001, 2, 15, 0, 0)
 datetime.datetime(2001, 3, 15, 0, 0) datetime.datetime(2001, 4, 15, 0, 0)
 datetime.datetime(2001, 5, 15, 0, 0) datetime.datetime(2001, 6, 15, 0, 0)
 datetime.datetime(2001, 7, 15, 0, 0) datetime.datetime(2001, 8, 15, 0, 0)
 datetime.datetime(2001, 9, 15, 0, 0)
 datetime.datetime(2001, 10, 15, 0, 0)
 datetime.datetime(2001, 11, 15, 0, 0)
 datetime.datetime(2001, 12, 15, 0, 0)]


In [71]:
dates_south = relative_doy_to_date(np.array([15, 46, 77, 107, 138, 168, 199, 230, 258, 289, 319, 350]), "south", 2001)
print(dates_south)

[datetime.datetime(2001, 7, 15, 0, 0) datetime.datetime(2001, 8, 15, 0, 0)
 datetime.datetime(2001, 9, 15, 0, 0)
 datetime.datetime(2001, 10, 15, 0, 0)
 datetime.datetime(2001, 11, 15, 0, 0)
 datetime.datetime(2001, 12, 15, 0, 0)
 datetime.datetime(2002, 1, 15, 0, 0) datetime.datetime(2002, 2, 15, 0, 0)
 datetime.datetime(2002, 3, 15, 0, 0) datetime.datetime(2002, 4, 15, 0, 0)
 datetime.datetime(2002, 5, 15, 0, 0) datetime.datetime(2002, 6, 15, 0, 0)]


In [72]:
# -----------------------------------------------------------
# Example vegetation period dates
# -----------------------------------------------------------

onset_date = np.empty(
    (2, 2),
    dtype=object
)

end_date = np.empty(
    (2, 2),
    dtype=object
)


# Pixel 0,0
onset_date[0, 0] = datetime(2000, 3, 15)
end_date[0, 0] = datetime(2001, 10, 20)

# Pixel 0,1
onset_date[0, 1] = datetime(2001, 4, 1)
end_date[0, 1] = datetime(2001, 9, 30)

# Pixel 1,0
onset_date[1, 0] = datetime(2001, 2, 20)
end_date[1, 0] = datetime(2001, 3, 10)

# Pixel 1,1: missing vegetation period
onset_date[1, 1] = None
end_date[1, 1] = None


# -----------------------------------------------------------
# Test March overlap
# -----------------------------------------------------------

march_fraction = month_overlap_fraction(
    month_year=2001,
    month=3,
    onset_date=onset_date,
    end_date=end_date
)


print(march_fraction)

[[1.         0.        ]
 [0.32258064 0.        ]]


In [73]:

year = 2001

onset_prev = np.array([
    [datetime(2000, 9, 15), datetime(2000, 12, 1)],
    [datetime(2000, 7, 1), datetime(2000, 11, 15)]
], dtype=object)

end_prev = np.array([
    [datetime(2001, 3, 20), datetime(2001, 1, 15)],
    [datetime(2001, 6, 30), datetime(2001, 2, 28)]
], dtype=object)

onset_current = np.array([
    [datetime(2001, 9, 1), datetime(2001, 7, 1)],
    [datetime(2001, 12, 15), datetime(2001, 8, 15)]
], dtype=object)

end_current = np.array([
    [datetime(2002, 2, 15), datetime(2002, 4, 1)],
    [datetime(2002, 3, 1), datetime(2002, 1, 31)]
], dtype=object)

veg_length = calculate_calendar_year_vegetation_length(
    onset_prev,
    end_prev,
    onset_current,
    end_current,
    year
)

print(veg_length)

[[201. 199.]
 [198. 198.]]


In [74]:

# -------------------------------------------------------
# Create temporary directory
# -------------------------------------------------------

tmpdir = tempfile.mkdtemp()

year = 2001

climate_files = []

profile = {
    "driver": "GTiff",
    "height": 1,
    "width": 1,
    "count": 3,
    "dtype": "int16",
    "crs": None,
    "transform": from_origin(0, 0, 1, 1),
    "nodata": -9999
}

# -------------------------------------------------------
# Create 12 monthly rasters
# -------------------------------------------------------

for month in range(1, 13):

    filename = os.path.join(
        tmpdir,
        f"Climate_monthly_{year}-{month:02d}-01.tif"
    )

    climate_files.append(filename)

    tmmn = np.array([[100 + 10*(month-1)]], dtype=np.int16)
    tmmx = np.array([[200 + 10*(month-1)]], dtype=np.int16)
    pr = np.array([[10*month]], dtype=np.int16)

    with rasterio.open(filename, "w", **profile) as dst:
        dst.write(tmmn, 1)
        dst.write(tmmx, 2)
        dst.write(pr, 3)

# -------------------------------------------------------
# Print values stored in monthly climate files
# -------------------------------------------------------

print("Input climate files\n")

for month, filename in enumerate(climate_files, start=1):

    with rasterio.open(filename) as src:

        tmmn_raw = src.read(1)[0, 0]
        tmmx_raw = src.read(2)[0, 0]
        pr = src.read(3)[0, 0]

        print(
            f"Month {month:02d}: "
            f"tmmn_raw={tmmn_raw:4d}, "
            f"tmmx_raw={tmmx_raw:4d}, "
            f"pr={pr:4d}"
        )

# -------------------------------------------------------
# Run aggregation
# -------------------------------------------------------

output_file = os.path.join(
    tmpdir,
    "Climate_yearly_2001.tif"
)

aggregate_climate_year(
    climate_files=climate_files,
    year=year,
    output_file=output_file
)

# -------------------------------------------------------
# Read result
# -------------------------------------------------------

with rasterio.open(output_file) as src:

    band_names = [
        "tmmn_mean",
        "tmmx_mean",
        "tmmn_min",
        "tmmn_max",
        "tmmx_min",
        "tmmx_max",
        "pr_sum"
    ]

    print("Results")

    for i, name in enumerate(band_names, start=1):
        print(
            f"{name:12s}:",
            src.read(i)[0, 0]
        )

# -------------------------------------------------------
# Calculate expected values
# -------------------------------------------------------

days = np.array([
    calendar.monthrange(year, m)[1]
    for m in range(1, 13)
])

tmmn = np.arange(10, 22)
tmmx = np.arange(20, 32)
pr = np.arange(10, 130, 10)

print("\nExpected")

print(
    "tmmn_mean :",
    np.sum(tmmn * days) / np.sum(days)
)

print(
    "tmmx_mean :",
    np.sum(tmmx * days) / np.sum(days)
)

print(
    "tmmn_min  :",
    np.min(tmmn)
)

print(
    "tmmn_max  :",
    np.max(tmmn)
)

print(
    "tmmx_min  :",
    np.min(tmmx)
)

print(
    "tmmx_max  :",
    np.max(tmmx)
)

print(
    "pr_sum    :",
    np.sum(pr)
)

Input climate files

Month 01: tmmn_raw= 100, tmmx_raw= 200, pr=  10
Month 02: tmmn_raw= 110, tmmx_raw= 210, pr=  20
Month 03: tmmn_raw= 120, tmmx_raw= 220, pr=  30
Month 04: tmmn_raw= 130, tmmx_raw= 230, pr=  40
Month 05: tmmn_raw= 140, tmmx_raw= 240, pr=  50
Month 06: tmmn_raw= 150, tmmx_raw= 250, pr=  60
Month 07: tmmn_raw= 160, tmmx_raw= 260, pr=  70
Month 08: tmmn_raw= 170, tmmx_raw= 270, pr=  80
Month 09: tmmn_raw= 180, tmmx_raw= 280, pr=  90
Month 10: tmmn_raw= 190, tmmx_raw= 290, pr= 100
Month 11: tmmn_raw= 200, tmmx_raw= 300, pr= 110
Month 12: tmmn_raw= 210, tmmx_raw= 310, pr= 120


/var/folders/v8/1_2rlmrd0wl5w5t5n60h8s440000gn/T/ipykernel_64295/1240463843.py:54: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  tmmn_raw = src.read(1)[0, 0]
/var/folders/v8/1_2rlmrd0wl5w5t5n60h8s440000gn/T/ipykernel_64295/1240463843.py:55: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  tmmx_raw = src.read(2)[0, 0]
/var/folders/v8/1_2rlmrd0wl5w5t5n60h8s440000gn/T/ipykernel_64295/1240463843.py:56: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  pr = src.read(3)[0, 0]


ValueError: Band 'tmmn' not found. Available bands: (None, None, None)

In [ ]:
# -------------------------------------------------------
# Create temporary folder
# -------------------------------------------------------

tmpdir = tempfile.mkdtemp()

year = 2001


# -------------------------------------------------------
# Create monthly climate rasters
# -------------------------------------------------------

climate_files = []

profile = {
    "driver": "GTiff",
    "height": 1,
    "width": 1,
    "count": 3,
    "dtype": "int16",
    "transform": from_origin(0, 0, 1, 1),
    "nodata": -9999
}


for month in range(1, 13):

    filename = os.path.join(
        tmpdir,
        f"Climate_monthly_{year}-{month:02d}-01.tif"
    )

    climate_files.append(filename)

    # raw values:
    # tmmn = 10°C, 11°C, ..., 21°C
    # tmmx = 20°C, 21°C, ..., 31°C
    # pr   = 10,20,...120 mm

    tmmn = np.array(
        [[100 + (month-1)*10]],
        dtype="int16"
    )

    tmmx = np.array(
        [[200 + (month-1)*10]],
        dtype="int16"
    )

    pr = np.array(
        [[month*10]],
        dtype="int16"
    )

    with rasterio.open(
        filename,
        "w",
        **profile
    ) as dst:

        dst.write(tmmn, 1)
        dst.write(tmmx, 2)
        dst.write(pr, 3)



# -------------------------------------------------------
# Create vegetation rasters
# -------------------------------------------------------

vegetation_files = []

veg_profile = profile.copy()
veg_profile.update(
    count=3,
    dtype="float32",
    nodata=np.nan
)


# Previous vegetation raster (2000)
# vegetation year: July 2000 - June 2001
#
# onset:
# 15 March 2001
# relative DOY:
# July 1 2000 = day 1
# March 15 2001 = day 258

onset_prev = np.array(
    [[258]],
    dtype="float32"
)

end_prev = np.array(
    [[365]],
    dtype="float32"
)

length_prev = np.array(
    [[108]],
    dtype="float32"
)


prev_file = os.path.join(
    tmpdir,
    "vegetation_2000.tif"
)

vegetation_files.append(prev_file)


with rasterio.open(
    prev_file,
    "w",
    **veg_profile
) as dst:

    dst.write(onset_prev, 1)
    dst.write(end_prev, 2)
    dst.write(length_prev, 3)



# Current vegetation raster (2001)
# vegetation year: July 2001 - June 2002
#
# onset:
# July 1 2001
# relative DOY = 1
#
# end:
# October 20 2001
# relative DOY = 112

onset_current = np.array(
    [[1]],
    dtype="float32"
)

end_current = np.array(
    [[112]],
    dtype="float32"
)

length_current = np.array(
    [[112]],
    dtype="float32"
)


current_file = os.path.join(
    tmpdir,
    "vegetation_2001.tif"
)

vegetation_files.append(current_file)


with rasterio.open(
    current_file,
    "w",
    **veg_profile
) as dst:

    dst.write(onset_current, 1)
    dst.write(end_current, 2)
    dst.write(length_current, 3)



# -------------------------------------------------------
# Run aggregation
# -------------------------------------------------------

output_file = os.path.join(
    tmpdir,
    "climate_vegetation_period_2001.tif"
)


aggregate_climate_veg_period(
    climate_files=climate_files,
    vegetation_files=vegetation_files,
    year=year,
    hemisphere="south",
    output_file=output_file
)



# -------------------------------------------------------
# Print output
# -------------------------------------------------------

band_names = [
    "veg_tmmn_mean",
    "veg_tmmx_mean",
    "veg_tmmn_min",
    "veg_tmmn_max",
    "veg_tmmx_min",
    "veg_tmmx_max",
    "veg_pr_sum",
    "vegetation_length"
]


with rasterio.open(output_file) as src:

    print("Output values\n")

    for i, name in enumerate(band_names, start=1):

        value = src.read(i)[0, 0]

        print(
            f"{name:20s}: {value}"
        )

TypeError: aggregate_climate_veg_period() missing 1 required positional argument: 'mask'